In [49]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from get_hr import get_hr

COLORS = {
    'green':     '#2ca02c',
    'accel_x':   '#1f77b4',
    'accel_y':   '#ff7f0e',
    'accel_z':   '#9467bd',
    'accel_xy':  '#17becf',
    'fused_mag': '#e377c2',
    'fused_pca': '#8c564b',
}

In [ ]:
csv_path = 'sleep/data/hardik@temple.com_2026-05-08T02:20:00_2026-05-08T09:15:30'
data = pd.read_csv(csv_path)
data['timestamp_ist'] = pd.to_datetime(data['timestamp'], unit='ms', utc=True).dt.tz_convert('Asia/Kolkata')
start = pd.Timestamp('2026-05-08 03:46:18', tz='Asia/Kolkata')
end = pd.Timestamp('2026-05-08 03:46:4', tz='Asia/Kolkata')
data = data[(data['timestamp_ist'] >= start) & (data['timestamp_ist'] < end)].reset_index(drop=True)

ppg_data = data.loc[data['green'].notna(), ['timestamp', 'green', 'red', 'ir']]
accel_data = data.loc[(data['accel_x'].notna() & data['accel_y'].notna() & data['accel_z'].notna()), ['timestamp', 'accel_x', 'accel_y', 'accel_z']]
tp = pd.to_datetime(ppg_data['timestamp'], unit='ms', utc=True).dt.tz_convert('Asia/Kolkata')
ta = pd.to_datetime(accel_data['timestamp'], unit='ms', utc=True).dt.tz_convert('Asia/Kolkata')

ppg_channels = ['green'] # 'red', 'ir'
accel_channels = ['accel_x', 'accel_y', 'accel_z']
raw_channels = ppg_channels + accel_channels
fig = make_subplots(rows=len(raw_channels), cols=1, shared_xaxes=True,
                    subplot_titles=raw_channels)
for i, name in enumerate(raw_channels, start=1):
    t = tp if name in ppg_channels else ta
    df = ppg_data if name in ppg_channels else accel_data
    fig.add_trace(go.Scatter(x=t, y=df[name], name=name,
                             line=dict(color=COLORS[name])),
                  row=i, col=1)
fig.update_xaxes(title_text='time (IST)', row=len(raw_channels), col=1)
fig.update_layout(height=600, showlegend=False,
                  title_text=f'Raw Signal')
fig.show()

In [51]:
from scipy.signal import butter, sosfiltfilt

TARGET_FS = 26.0

RESP_BAND_HZ          = (0.1, 0.5)   # 6–30 brpm

SOS_RESP_BP     = butter(4, list(RESP_BAND_HZ),    btype='bandpass', fs=TARGET_FS, output='sos')
SOS_RESP_PPG    = butter(4, list(RESP_BAND_HZ),    btype='bandpass', fs=TARGET_FS, output='sos')


def preprocess(sig: np.ndarray) -> np.ndarray:
    sig = sig - np.mean(sig)                       # DC
    return sosfiltfilt(SOS_RESP_BP, sig)           # respiration band

In [52]:
# Resample PPG (32 Hz) onto the accel sample times (26 Hz) so len(green) == len(accel_*).
t_ppg_ms   = ppg_data['timestamp'].to_numpy()
t_accel_ms = accel_data['timestamp'].to_numpy()

green   = np.interp(t_accel_ms, t_ppg_ms, ppg_data['green'].to_numpy())
accel_x = accel_data['accel_x'].to_numpy()
accel_y = accel_data['accel_y'].to_numpy()
accel_z = accel_data['accel_z'].to_numpy()

t_rs = t_accel_ms
ga_raw = pd.DataFrame({
    'timestamp': t_rs,
    'green': green,
    'accel_x': accel_x,
    'accel_y': accel_y,
    'accel_z': accel_z,
})
assert len(green) == len(accel_x) == len(accel_y) == len(accel_z) == len(accel_data)

t = pd.to_datetime(t_rs, unit='ms', utc=True).tz_convert('Asia/Kolkata')
cols = ['green', 'accel_x', 'accel_y', 'accel_z']
fig = make_subplots(rows=len(cols), cols=1, shared_xaxes=True, subplot_titles=cols)
for i, c in enumerate(cols, start=1):
    fig.add_trace(
        go.Scatter(x=t, y=ga_raw[c], name=c,
                   line=dict(color=COLORS[c], width=1)),
        row=i, col=1,
    )
fig.update_xaxes(title_text='time (IST)', row=len(cols), col=1)
fig.update_layout(height=180 * len(cols), showlegend=False,
                  title_text=f'Resampled PPG')
fig.show()

In [53]:
green = sosfiltfilt(SOS_RESP_PPG, ga_raw['green'])
accel_x = preprocess(ga_raw['accel_x'])
accel_y = preprocess(ga_raw['accel_y'])
accel_z = preprocess(ga_raw['accel_z'])

ga_filt = pd.DataFrame({
    'timestamp': t_rs,
    'green': green,
    'accel_x': accel_x,
    'accel_y': accel_y,
    'accel_z': accel_z,
})

cols = [c for c in ga_filt.columns if c != 'timestamp']
fig = make_subplots(rows=len(cols), cols=1, shared_xaxes=True, subplot_titles=cols)
for i, c in enumerate(cols, start=1):
    fig.add_trace(
        go.Scatter(x=t, y=ga_filt[c], name=c,
                   line=dict(color=COLORS[c], width=1)),
        row=i, col=1,
    )
fig.update_xaxes(title_text='time (IST)', row=len(cols), col=1)
fig.update_layout(height=180 * len(cols), showlegend=False,
                  title_text=f'Bandpassed (respiration band)')
fig.show()

In [54]:
def fusion_magnitude(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    return np.sqrt(x ** 2 + y ** 2)

def fusion_pca(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """First principal component of [x, y] — axis with most variance."""
    M = np.column_stack([x, y])
    cov = (M.T @ M) / max(len(M) - 1, 1)
    w, V = np.linalg.eigh(cov)
    pc1 = V[:, np.argmax(w)]
    return M @ pc1

fused = {
    'fused_mag': fusion_magnitude(accel_x, accel_y),
    'fused_pca': fusion_pca(accel_x, accel_y),
}

t_fused = pd.to_datetime(t_rs, unit='ms', utc=True).tz_convert('Asia/Kolkata')
fig = make_subplots(rows=len(fused), cols=1, shared_xaxes=True,
                    subplot_titles=('mag', 'pca'))
for i, (name, sig) in enumerate(fused.items(), start=1):
    fig.add_trace(go.Scatter(x=t_fused, y=sig, name=name,
                             line=dict(color=COLORS[name], width=1)),
                  row=i, col=1)
fig.update_xaxes(title_text='time (IST)', row=len(fused), col=1)
fig.update_layout(height=540, showlegend=False,
                  title_text=f'Fused')
fig.show()

/var/folders/bf/vjlr28hn7gqcbr_213rvzph80000gn/T/ipykernel_6818/1284690580.py:10: RuntimeWarning: divide by zero encountered in matmul
  return M @ pc1
/var/folders/bf/vjlr28hn7gqcbr_213rvzph80000gn/T/ipykernel_6818/1284690580.py:10: RuntimeWarning: overflow encountered in matmul
  return M @ pc1
/var/folders/bf/vjlr28hn7gqcbr_213rvzph80000gn/T/ipykernel_6818/1284690580.py:10: RuntimeWarning: invalid value encountered in matmul
  return M @ pc1


In [55]:
FFT_SIZE = 1024
RESP_MIN_BRPM = 6.0
RESP_MAX_BRPM = 30.0
F_LO = RESP_MIN_BRPM / 60.0
F_HI = RESP_MAX_BRPM / 60.0


def _padded(x: np.ndarray) -> tuple[np.ndarray, int]:
    """Zero-pad x into a length-FFT_SIZE buffer. Returns (buf, n_used)."""
    buf = np.zeros(FFT_SIZE, dtype=np.float64)
    n = min(len(x), FFT_SIZE)
    buf[:n] = x[:n]
    return buf, n


def rfft_magnitude(x: np.ndarray) -> tuple[np.ndarray, float]:
    """rFFT magnitude with linear ramp window. Returns (mags, freq_res_hz)."""
    buf, n = _padded(x)
    if n >= 2:
        buf[:n] *= 0.5 + 0.5 * np.arange(n) / (n - 1)
    mags = np.abs(np.fft.rfft(buf))
    freq_res = TARGET_FS / FFT_SIZE
    return mags, freq_res


def find_dominant_peak(mags: np.ndarray, freq_res: float) -> float:
    """Argmax of mags over [F_LO, F_HI]. Returns freq in Hz."""
    k_lo = max(0, int(F_LO / freq_res))
    k_hi = min(len(mags) - 1, int(F_HI / freq_res))
    if k_hi < k_lo:
        return 0.0
    k = int(np.argmax(mags[k_lo:k_hi + 1])) + k_lo
    return k * freq_res


def autocorrelation(x: np.ndarray) -> tuple[np.ndarray, float]:
    """Linear ACF via zero-padded FFT. Returns (acf[:n], lag_step_s)."""
    buf, n = _padded(x)
    lag_step = 1.0 / TARGET_FS
    if n == 0:
        return np.zeros(0, dtype=np.float64), lag_step
    spec = np.fft.rfft(buf)
    acf = np.fft.irfft(spec * np.conj(spec), n=FFT_SIZE)[:n]
    if acf[0] > 0:
        acf = acf / acf[0]
    return acf, lag_step


def pick_autocorrelation(acf: np.ndarray, lag_step: float) -> float:
    """Argmax of acf over lags [1/F_HI, 1/F_LO]. Returns freq in Hz."""
    n = len(acf)
    if n < 2:
        return 0.0
    k_lo = max(1, int((1.0 / F_HI) / lag_step))
    k_hi = min(n - 1, int((1.0 / F_LO) / lag_step))
    if k_hi <= k_lo:
        return 0.0
    k = int(np.argmax(acf[k_lo:k_hi + 1])) + k_lo
    period_s = k * lag_step
    return 1.0 / period_s if period_s > 0 else 0.0

In [56]:
# raw (green, accel_*):    rfft + dominant peak
# fused_mag/_pca:          autocorrelation

fft_channels = {
    'green':   ga_filt['green'].to_numpy(),
    'accel_x': ga_filt['accel_x'].to_numpy(),
    'accel_y': ga_filt['accel_y'].to_numpy(),
    'accel_z': ga_filt['accel_z'].to_numpy(),
}
acf_channels = {
    'fused_mag': fused['fused_mag'],
    'fused_pca': fused['fused_pca'],
}

specs = {name: rfft_magnitude(sig) for name, sig in fft_channels.items()}
acfs  = {name: autocorrelation(sig) for name, sig in acf_channels.items()}

fft_peaks_brpm = {
    name: find_dominant_peak(mags, fr) * 60.0
    for name, (mags, fr) in specs.items()
}
acf_peaks_brpm = {
    name: pick_autocorrelation(acf, ls) * 60.0
    for name, (acf, ls) in acfs.items()
}

# Shared brpm axes
freq_res = next(iter(specs.values()))[1]
fft_half = FFT_SIZE // 2 + 1
freqs_brpm_fft = np.arange(fft_half) * freq_res * 60.0
band_fft = (freqs_brpm_fft >= RESP_MIN_BRPM) & (freqs_brpm_fft <= RESP_MAX_BRPM)

lag_step_acf = next(iter(acfs.values()))[1]
n_lags = len(next(iter(acfs.values()))[0])
lags = np.arange(1, n_lags) * lag_step_acf
lags_brpm_acf = 60.0 / lags
band_acf = (lags_brpm_acf >= RESP_MIN_BRPM) & (lags_brpm_acf <= RESP_MAX_BRPM)

ordered = ['green', 'accel_x', 'accel_y', 'accel_z', 'fused_mag', 'fused_pca']
titles = {
    'green':     'green spectrum (FFT)',
    'accel_x':   'accel_x spectrum (FFT)',
    'accel_y':   'accel_y spectrum (FFT)',
    'accel_z':   'accel_z spectrum (FFT)',
    'fused_mag': 'fused_mag ACF',
    'fused_pca': 'fused_pca ACF',
}

fig = make_subplots(rows=len(ordered), cols=1, shared_xaxes=True,
                    subplot_titles=[titles[n] for n in ordered])

def _mark(row, x_brpm, y_max, color):
    fig.add_vline(x=x_brpm, line_dash='dash', line_color=color, row=row, col=1)
    fig.add_annotation(x=x_brpm, y=y_max,
                       text=f'{x_brpm:.1f}',
                       font=dict(color=color),
                       showarrow=True, arrowhead=2,
                       ax=60, ay=-25, row=row, col=1)

for row, name in enumerate(ordered, start=1):
    color = COLORS[name]

    if name in fft_channels:
        mags = specs[name][0]
        y_band = mags[band_fft]
        if not len(y_band):
            continue
        y_max = float(y_band.max())
        fig.add_trace(go.Scatter(x=freqs_brpm_fft[band_fft], y=y_band,
                                 mode='lines',
                                 line=dict(color=color)),
                      row=row, col=1)
        peak_brpm = fft_peaks_brpm[name]
    else:
        acf, _ = acfs[name]
        y_band = acf[1:][band_acf]
        if not len(y_band):
            continue
        y_max = float(y_band.max())
        fig.add_trace(go.Scatter(x=lags_brpm_acf[band_acf], y=y_band,
                                 mode='lines',
                                 line=dict(color=color)),
                      row=row, col=1)
        peak_brpm = acf_peaks_brpm[name]

    if peak_brpm > 0:
        _mark(row, peak_brpm, y_max, color)

fig.update_xaxes(title_text='brpm', row=len(ordered), col=1)
fig.update_layout(height=160 * len(ordered), showlegend=False,
                  title_text='Per-channel respiration peaks (raw: FFT, fused: ACF)')
fig.show()